# Phase 3: Error Analysis
## DNA Gene Mapping Project
**Author:** Sharique Mohammad  
**Date:** February 2026  

---

## Objective
Deep dive into model errors:
- Analyze false positives and false negatives
- Identify patterns in misclassifications
- Understand which variant types are problematic
- Provide insights for model improvement

## Why Error Analysis Matters
Understanding errors helps:
- Identify model weaknesses
- Guide feature engineering improvements
- Set appropriate decision thresholds
- Communicate limitations to users

---
## 1. Setup

In [1]:
import pandas as pd
import numpy as np
import pickle
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns

import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')

PROJECT_ROOT = Path.cwd().parent.parent
DATA_DIR = PROJECT_ROOT / "data" / "ml"
MODEL_DIR = PROJECT_ROOT / "models"
FIGURES_DIR = PROJECT_ROOT / "data" / "analytical" / "figures" / "phase3"
REPORTS_DIR = PROJECT_ROOT / "data" / "analytical" / "reports"

print("="*80)
print("ERROR ANALYSIS")
print("="*80)

ERROR ANALYSIS


---
## 2. Load Data and Models

In [2]:
# Load test data
with open(DATA_DIR / "variant_test.pkl", 'rb') as f:
    test_data = pickle.load(f)
    X_test, y_test = test_data['X'], test_data['y']

# Load model
with open(MODEL_DIR / "ensemble_xgb_variants.pkl", 'rb') as f:
    model = pickle.load(f)

print(f"Test set: {X_test.shape}")
print(f"Features: {list(X_test.columns[:10])}...")  # Show first 10

Test set: (623435, 75)
Features: ['is_constrained', 'gene_has_omim', 'review_quality_score', 'is_mitochondrial_variant', 'gene_vus_ratio', 'is_receptor', 'is_cardiovascular_gene_variant', 'has_omim_disease', 'has_functional_domain', 'gene_avg_druggability']...


---
## 3. Generate Predictions and Identify Errors

In [3]:
# Make predictions
y_pred = model.predict(X_test)
y_proba = model.predict_proba(X_test)[:, 1]

# Identify error types
true_negatives = (y_test == False) & (y_pred == 0)
false_positives = (y_test == False) & (y_pred == 1)
false_negatives = (y_test == True) & (y_pred == 0)
true_positives = (y_test == True) & (y_pred == 1)

print("\n" + "="*80)
print("ERROR BREAKDOWN")
print("="*80)
print(f"\nTrue Negatives:  {true_negatives.sum():,}")
print(f"False Positives: {false_positives.sum():,} (Benign predicted as Pathogenic)")
print(f"False Negatives: {false_negatives.sum():,} (Pathogenic predicted as Benign)")
print(f"True Positives:  {true_positives.sum():,}")

print(f"\nTotal Errors: {(false_positives.sum() + false_negatives.sum()):,}")
print(f"Error Rate: {(false_positives.sum() + false_negatives.sum()) / len(y_test):.2%}")


ERROR BREAKDOWN

True Negatives:  562,918
False Positives: 11,246 (Benign predicted as Pathogenic)
False Negatives: 5,778 (Pathogenic predicted as Benign)
True Positives:  43,493

Total Errors: 17,024
Error Rate: 2.73%


---
## 4. False Positive Analysis

In [4]:
print("\n" + "="*80)
print("FALSE POSITIVE ANALYSIS")
print("="*80)

# Extract false positives
fp_features = X_test[false_positives]
fp_probabilities = y_proba[false_positives]

print(f"\nAnalyzing {len(fp_features):,} false positives...")

# Probability distribution of false positives
print(f"\nProbability Statistics:")
print(f"  Mean: {fp_probabilities.mean():.3f}")
print(f"  Median: {np.median(fp_probabilities):.3f}")
print(f"  Min: {fp_probabilities.min():.3f}")
print(f"  Max: {fp_probabilities.max():.3f}")

# High confidence false positives (probability > 0.8)
high_conf_fp = fp_probabilities > 0.8
print(f"\nHigh confidence FP (prob > 0.8): {high_conf_fp.sum():,} ({high_conf_fp.sum()/len(fp_probabilities)*100:.1f}%)")

# Top features in false positives
print(f"\nTop 5 Feature Means (False Positives):")
for col in ['is_high_impact', 'gene_has_omim', 'pathogenicity_score', 'phylop_score', 'cadd_phred']:
    if col in fp_features.columns:
        print(f"  {col}: {fp_features[col].mean():.3f}")


FALSE POSITIVE ANALYSIS

Analyzing 11,246 false positives...

Probability Statistics:
  Mean: 0.726
  Median: 0.712
  Min: 0.500
  Max: 0.999

High confidence FP (prob > 0.8): 3,927 (34.9%)

Top 5 Feature Means (False Positives):
  is_high_impact: 0.974
  gene_has_omim: -0.048
  pathogenicity_score: 0.876
  phylop_score: 0.616
  cadd_phred: -0.069


---
## 5. False Negative Analysis

In [5]:
print("\n" + "="*80)
print("FALSE NEGATIVE ANALYSIS")
print("="*80)

# Extract false negatives
fn_features = X_test[false_negatives]
fn_probabilities = y_proba[false_negatives]

print(f"\nAnalyzing {len(fn_features):,} false negatives...")

# Probability distribution
print(f"\nProbability Statistics:")
print(f"  Mean: {fn_probabilities.mean():.3f}")
print(f"  Median: {np.median(fn_probabilities):.3f}")
print(f"  Min: {fn_probabilities.min():.3f}")
print(f"  Max: {fn_probabilities.max():.3f}")

# Low confidence false negatives (probability < 0.2)
low_conf_fn = fn_probabilities < 0.2
print(f"\nLow confidence FN (prob < 0.2): {low_conf_fn.sum():,} ({low_conf_fn.sum()/len(fn_probabilities)*100:.1f}%)")

# Top features in false negatives
print(f"\nTop 5 Feature Means (False Negatives):")
for col in ['is_high_impact', 'gene_has_omim', 'pathogenicity_score', 'phylop_score', 'cadd_phred']:
    if col in fn_features.columns:
        print(f"  {col}: {fn_features[col].mean():.3f}")


FALSE NEGATIVE ANALYSIS

Analyzing 5,778 false negatives...

Probability Statistics:
  Mean: 0.232
  Median: 0.223
  Min: 0.000
  Max: 0.500

Low confidence FN (prob < 0.2): 2,661 (46.1%)

Top 5 Feature Means (False Negatives):
  is_high_impact: 0.154
  gene_has_omim: -0.085
  pathogenicity_score: 0.283
  phylop_score: 0.450
  cadd_phred: 0.012


---
## 6. Feature Distribution Comparison

In [6]:
print("\nGenerating error pattern visualizations...")

# Compare feature distributions: True Positives vs False Negatives
key_features = ['is_high_impact', 'gene_has_omim', 'pathogenicity_score', 'phylop_score']

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

tp_features = X_test[true_positives]

for idx, feature in enumerate(key_features):
    if feature in X_test.columns:
        ax = axes[idx]
        
        # Plot distributions
        ax.hist(tp_features[feature], bins=30, alpha=0.5, label='True Positives', density=True)
        ax.hist(fn_features[feature], bins=30, alpha=0.5, label='False Negatives', density=True)
        
        ax.set_xlabel(feature)
        ax.set_ylabel('Density')
        ax.legend()
        ax.grid(True, alpha=0.3)

plt.suptitle('Feature Distributions: True Positives vs False Negatives', fontsize=14)
plt.tight_layout()
plt.savefig(FIGURES_DIR / "33_error_feature_distributions.png", dpi=150, bbox_inches='tight')
plt.close()

print("OK Error pattern plot saved")


Generating error pattern visualizations...
OK Error pattern plot saved


---
## 7. Probability Threshold Analysis

In [7]:
print("\n" + "="*80)
print("THRESHOLD SENSITIVITY ANALYSIS")
print("="*80)

# Test different thresholds
thresholds = [0.3, 0.4, 0.5, 0.6, 0.7]

results = []
for thresh in thresholds:
    y_pred_thresh = (y_proba >= thresh).astype(int)
    
    tp = ((y_test == True) & (y_pred_thresh == 1)).sum()
    fp = ((y_test == False) & (y_pred_thresh == 1)).sum()
    fn = ((y_test == True) & (y_pred_thresh == 0)).sum()
    tn = ((y_test == False) & (y_pred_thresh == 0)).sum()
    
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0
    
    results.append({
        'threshold': thresh,
        'TP': tp,
        'FP': fp,
        'FN': fn,
        'TN': tn,
        'precision': precision,
        'recall': recall,
        'f1': f1
    })

threshold_df = pd.DataFrame(results)
print("\n" + threshold_df.to_string(index=False))

# Find best threshold
best_thresh = threshold_df.loc[threshold_df['f1'].idxmax()]
print(f"\nBest Threshold: {best_thresh['threshold']} (F1={best_thresh['f1']:.4f})")


THRESHOLD SENSITIVITY ANALYSIS

 threshold    TP    FP   FN     TN  precision   recall       f1
       0.3 45631 20996 3640 553168   0.684872 0.926123 0.787434
       0.4 44652 15307 4619 558857   0.744709 0.906253 0.817578
       0.5 43493 11246 5778 562918   0.794552 0.882730 0.836323
       0.6 42292  8203 6979 565961   0.837548 0.858355 0.847824
       0.7 40811  5879 8460 568285   0.874084 0.828297 0.850575

Best Threshold: 0.7 (F1=0.8506)


---
## 8. Save Error Analysis Report

In [8]:
# Create error analysis report
error_report = {
    'total_samples': len(y_test),
    'total_errors': int(false_positives.sum() + false_negatives.sum()),
    'error_rate': float((false_positives.sum() + false_negatives.sum()) / len(y_test)),
    'false_positives': {
        'count': int(false_positives.sum()),
        'mean_probability': float(fp_probabilities.mean()),
        'high_confidence_count': int((fp_probabilities > 0.8).sum())
    },
    'false_negatives': {
        'count': int(false_negatives.sum()),
        'mean_probability': float(fn_probabilities.mean()),
        'low_confidence_count': int((fn_probabilities < 0.2).sum())
    },
    'best_threshold': {
        'threshold': float(best_thresh['threshold']),
        'f1_score': float(best_thresh['f1']),
        'precision': float(best_thresh['precision']),
        'recall': float(best_thresh['recall'])
    }
}

import json
with open(REPORTS_DIR / "error_analysis.json", 'w') as f:
    json.dump(error_report, f, indent=2)

print("\nOK Error analysis saved: error_analysis.json")


OK Error analysis saved: error_analysis.json


---
## 9. Summary and Recommendations

In [9]:
print("\n" + "="*80)
print("ERROR ANALYSIS SUMMARY")
print("="*80)

print(f"\nTotal Errors: {false_positives.sum() + false_negatives.sum():,}")
print(f"Error Rate: {(false_positives.sum() + false_negatives.sum()) / len(y_test):.2%}")

print(f"\nFalse Positives: {false_positives.sum():,}")
print(f"  - Benign variants incorrectly flagged as pathogenic")
print(f"  - Mean probability: {fp_probabilities.mean():.3f}")
print(f"  - High confidence errors (>0.8): {(fp_probabilities > 0.8).sum():,}")

print(f"\nFalse Negatives: {false_negatives.sum():,}")
print(f"  - Pathogenic variants missed by the model")
print(f"  - Mean probability: {fn_probabilities.mean():.3f}")
print(f"  - Low confidence errors (<0.2): {(fn_probabilities < 0.2).sum():,}")

print("\nRECOMMENDATIONS:")
if false_negatives.sum() > false_positives.sum():
    print("  - Consider lowering threshold to catch more pathogenic variants")
    print("  - Add more features related to pathogenicity signals")
else:
    print("  - Consider raising threshold to reduce false alarms")
    print("  - Review features that cause benign variants to score high")

print("\n" + "="*80)
print("FILES CREATED")
print("="*80)
print("\nFigures:")
print("  - 33_error_feature_distributions.png")
print("\nReports:")
print("  - error_analysis.json")

print("\n" + "="*80)
print("ERROR ANALYSIS COMPLETE")
print("="*80)


ERROR ANALYSIS SUMMARY

Total Errors: 17,024
Error Rate: 2.73%

False Positives: 11,246
  - Benign variants incorrectly flagged as pathogenic
  - Mean probability: 0.726
  - High confidence errors (>0.8): 3,927

False Negatives: 5,778
  - Pathogenic variants missed by the model
  - Mean probability: 0.232
  - Low confidence errors (<0.2): 2,661

RECOMMENDATIONS:
  - Consider raising threshold to reduce false alarms
  - Review features that cause benign variants to score high

FILES CREATED

Figures:
  - 33_error_feature_distributions.png

Reports:
  - error_analysis.json

ERROR ANALYSIS COMPLETE
